# Preprocessing of NTCIR-2 Adhoc Dataset

Note: This preprocessing is designd for [geniie-lab](https://github.com/geniie-lab/geniie-lab) experiments and not necessarily suitable for other tasks.

## Data path
- Get a copy of the dataset (see [README.md](README.md)).
- We assume that the downloaded file has been uncompressed to the following path.

In [ ]:
import os
os.environ['DATA1'] = 'path to your ntcir-1 data folder'
os.environ['DATA2'] = 'path to your ntcir-2 data folder'

In [ ]:
!ls $DATA2

## Preprocessing of corpus files

- NTCIR-2 uses both the new corpus files and that of NTCIR-1

In [ ]:
!tar xvfz $DATA2/j-docs.tgz -C $DATA2/

In [ ]:
!iconv -f EUC-JP -t UTF-8 -c $DATA2/j-docs/ntc2-j1g > $DATA2/j-docs/ntc2-j1g.utf8
!iconv -f EUC-JP -t UTF-8 -c $DATA2/j-docs/ntc2-j1k > $DATA2/j-docs/ntc2-j1k.utf8

In [ ]:
# Number of documents
!grep "^<ACCN" $DATA2/j-docs/ntc2-j1*.utf8 | wc -l

### ntc2-j1g

In [ ]:
import sys
import re
import json
def docs_g_jsonl(in_file):
    out_file = in_file + '.jsonl'
    with open(in_file, 'r') as f, open(out_file, 'w') as f1:
        record = ''
        items = {}
        count = 0
        for line in f:
            line = line.rstrip()
            if line == '</REC>':
                accn = re.findall(r'<ACCN>(.+?)<', record)[0]
                titl = re.findall(r'<TITL .+?>(.+?)<', record)[0]
                abst = re.findall(r'<ABST .+?>(.+?)</ABST>', record)[0]
                abst = re.sub(r'<ABST.P>', '', abst)
                abst = re.sub(r'</ABST.P>', '', abst)
                contents = titl + ' ' + abst
                items = {
                    'doc_id': accn,
                    'text': contents
                }
                j = json.dumps(items, ensure_ascii=False)
                f1.write(f'{j}\n')
                record = ''
                items = {}
                count += 1
                if count % 10000 == 0:
                    print(f'{count}, ', end='', file=sys.stderr)
            else:
                record += line
        print(f'{count}, Done!', file=sys.stderr)

In [ ]:
docs_g_jsonl(os.getenv('DATA2') + '/j-docs/ntc2-j1g.utf8')

### ntc2-j1k

In [ ]:
import sys
import re
import json
def docs_k_jsonl(in_file):
    out_file = in_file + '.jsonl'
    with open(in_file, 'r') as f, open(out_file, 'w') as f1:
        record = ''
        items = {}
        count = 0
        for line in f:
            line = line.rstrip()
            if line == '</REC>':
                accn = re.findall(r'<ACCN>(.+?)<', record)[0]
                titl = re.findall(r'<PJNM .+?>(.+?)<', record)[0] # Difference
                abst = re.findall(r'<ABST .+?>(.+?)</ABST>', record)[0]
                abst = re.sub(r'<ABST.P>', '', abst)
                abst = re.sub(r'</ABST.P>', '', abst)
                contents = titl + ' ' + abst
                items = {
                    'doc_id': accn,
                    'text': contents
                }
                j = json.dumps(items, ensure_ascii=False)
                f1.write(f'{j}\n')
                record = ''
                items = {}
                count += 1
                if count % 10000 == 0:
                    print(f'{count}, ', end='', file=sys.stderr)
            else:
                record += line
        print(f'{count}, Done!', file=sys.stderr)

In [ ]:
docs_k_jsonl(os.getenv('DATA2') + '/j-docs/ntc2-j1k.utf8')

In [ ]:
!wc -l $DATA2/j-docs/ntc2-j1*.utf8.jsonl

### NTCIR-1 corpus file

In [ ]:
import sys
import json
def convert_ntcir1_to_ntcir2(in_file, out_file):
    with open(in_file, 'r') as f, open(out_file, 'w') as f2:
        for i, line in enumerate(f):
            j = json.loads(line)
            docid = j['doc_id'].replace('gakkai-', 'gakkai-j-')
            j['doc_id'] = docid
            jline = json.dumps(j, ensure_ascii=False)
            f2.write(f'{jline}\n')
            if i % 10000 == 0:
                print(f'{i}, ', end='', file=sys.stderr)
        print(f'{i}, Done!', file=sys.stderr)

In [ ]:
convert_ntcir1_to_ntcir2(
    os.getenv('DATA1') + '/mlir/ntc1-j1.utf8.jsonl',
    os.getenv('DATA2') + '/j-docs/ntc1-j1.utf8.mod.jsonl'
)

In [ ]:
!cat $DATA2/j-docs/ntc1-j1.utf8.mod.jsonl $DATA2/j-docs/ntc2-j1g.utf8.jsonl $DATA2/j-docs/ntc2-j1k.utf8.jsonl > $DATA2/j-docs/ntc12-j1gk.mod.jsonl

In [ ]:
!ls $DATA2/j-docs

## Topic files

In [ ]:
!tar xvfz $DATA2/topics.tgz -C $DATA2/

In [ ]:
!iconv -f EUC-JP -t UTF-8 -c $DATA2/topics/topic-j0101-0149 > $DATA2/topics/topic-j0101-0149.utf8

In [ ]:
!ls $DATA2/topics

In [ ]:
import re
def topics_jsonl(in_file):
    out_file = in_file + '.jsonl'
    with open(in_file, 'r') as f:
        s = f.read()
        qid = re.findall('<TOPIC q=([^>]+)>', s)
        title = re.findall('<TITLE>\n(.*)\n</TITLE>', s)
        desc = re.findall('<DESCRIPTION>\n(.*)\n</DESCRIPTION>', s)
        narr = re.findall('<NARRATIVE>\n(.*)\n</NARRATIVE>', s)
    with open(out_file, 'w') as f:
        for i in range(len(qid)):
            f.write(f'{{ "query_id": "{qid[i]}", "text": "{title[i]}", "description": "{desc[i]}", "narrative": "{narr[i]}" }}\n')

In [ ]:
topics_jsonl(os.getenv('DATA2') + '/topics/topic-j0101-0149.utf8')

In [ ]:
import re
def ntcir_to_trec(in_file):
    out_file = in_file + '.trec'

    with open(in_file, 'r', encoding='utf-8') as f:
        s = f.read()

        qid = re.findall(r'<TOPIC q=([^>]+)>', s)
        title = re.findall(r'<TITLE>\s*(.*?)\s*</TITLE>', s, re.DOTALL)
        desc = re.findall(r'<DESCRIPTION>\s*(.*?)\s*</DESCRIPTION>', s, re.DOTALL)
        narr = re.findall(r'<NARRATIVE>\s*(.*?)\s*</NARRATIVE>', s, re.DOTALL)

    with open(out_file, 'w', encoding='utf-8') as f:
        for i in range(len(qid)):
            f.write(
f"""<top>
<num> Number: {qid[i]}
<title> {title[i].strip()}

<desc> Description:
{desc[i].strip()}

<narr> Narrative:
{narr[i].strip()}
</top>

"""
            )

In [ ]:
ntcir_to_trec(os.getenv('DATA2') + '/topics/topic-j0101-0149.utf8')

In [ ]:
!ls $DATA2/topics

### Qrel files
- This test collection provides graded relevance scores (A: Relevant, B: Partially Relevant, C: Not Relevant)
- We convert them as follows.
    - A: 2
    - B: 1
    - C: 0

In [ ]:
!tar xvfz $DATA2/rels.tgz -C $DATA2/

In [ ]:
!iconv -f EUC-JP -t UTF-8 -c $DATA2/rels/rel2_ntc2-j2_0101-0149.nc > $DATA2/rels/rel2_ntc2-j2_0101-0149.nc.utf8

In [ ]:
def qrel_graded_tsv(in_file):
    out_file = in_file + '.tsv'
    with open(in_file, 'r') as f, open(out_file, 'w') as f2:
        for line in f:
            line = line.rstrip()
            flds = line.split('\t')
            if flds[1] == 'A':
                f2.write(f'{flds[0]}\tQ0\t{flds[2]}\t2\n')
            if flds[1] == 'B':
                f2.write(f'{flds[0]}\tQ0\t{flds[2]}\t1\n')
            if flds[1] == 'C':
                f2.write(f'{flds[0]}\tQ0\t{flds[2]}\t0\n')

In [ ]:
qrel_graded_tsv(os.getenv('DATA2') + '/rels/rel2_ntc2-j2_0101-0149.nc.utf8')

In [ ]:
!ls $DATA2/rels/

## Register to ir_datasets module locally

- Dataset name: `ntcir2-adhoc`

In [ ]:
# Remove old cache (if any)
!rm -rf ~/.ir_datasets/ntcir2-adhoc

In [ ]:
# Copy files
!mkdir -p ~/.ir_datasets/ntcir2-adhoc
!cp $DATA2/j-docs/ntc12-j1gk.mod.jsonl ~/.ir_datasets/ntcir2-adhoc/
!cp $DATA2/rels/rel2_ntc2-j2_0101-0149.nc.utf8.tsv ~/.ir_datasets/ntcir2-adhoc/
!cp $DATA2/topics/topic-j0101-0149.utf8.trec ~/.ir_datasets/ntcir2-adhoc/

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas

In [ ]:
import os
sys.path.append(os.path.join(os.getcwd()))

In [ ]:
import ir_datasets
import ntcir2_adhoc
dataset = ir_datasets.load('ntcir2-adhoc')
docstore = dataset.docs_store()
docstore.build()

In [ ]:
dataset.docs_cls().__annotations__

In [ ]:
docstore = dataset.docs_store()
docstore.get('kaken-j-0924516300').text # the one in the overview paper

In [ ]:
dataset.queries_cls().__annotations__

In [ ]:
import pandas as pd
pd.DataFrame(dataset.queries_iter())

In [ ]:
dataset.qrels_defs()

In [ ]:
pd.DataFrame(dataset.qrels_iter())